In [ ]:
import pandas as pd
import numpy as np
import igraph as ip
import networkx as nx
import matplotlib.pyplot as plt
from fpdf import FPDF
import seaborn as sns
from scipy import stats
import os
import heapq
import math
import pickle

### Data

In [ ]:
data_s2 = pd.read_csv("../data/science.add9330_data_s2.csv", header=0)

In [ ]:
coord = pd.read_csv("../emisferi/larva/csv/Coordinate/Coordinate 2.0.csv", header=0)

In [ ]:
adj_matrix = pd.read_csv("../data/all-all_connectivity_matrix.csv", header=[0], index_col=[0])
adj_matrix = adj_matrix.map(lambda x: 1 if x != 0 else 0)

# Connectome C & Distance Network D

## Costruzione

### Connectome

In [ ]:
connectome = nx.DiGraph(np.array(adj_matrix))
connectome = nx.relabel_nodes(connectome, dict(enumerate(adj_matrix.index)))

In [ ]:
for n, data in connectome.nodes(data=True):
    found = False
    try:
        data['celltype'] = data_s2[data_s2['left_id'] == str(n)]['celltype'].iloc[0]
        data['additional_annotations'] = data_s2[data_s2['left_id'] == str(n)]['additional_annotations'].iloc[0]
        data['level_7_cluster'] = data_s2[data_s2['left_id'] == str(n)]['level_7_cluster'].iloc[0]
        data['hemisphere'] = 'right'
        found = True
    except IndexError:
        pass
    if not found:
        try:
            data['celltype'] = data_s2[data_s2['right_id'] == str(n)]['celltype'].iloc[0]
            data['additional_annotations'] = data_s2[data_s2['right_id'] == str(n)]['additional_annotations'].iloc[0]
            data['level_7_cluster'] = data_s2[data_s2['right_id'] == str(n)]['level_7_cluster'].iloc[0]
            data['hemisphere'] = 'left'
        except IndexError:
            data['celltype'] = 'undefined'
            data['additional_annotations'] = 'undefined'
            data['level_7_cluster'] = 'undefined'
            data['hemisphere'] = 'undefined'
    node_coord = coord[coord['id']==n].iloc[0]
    data['x'] = node_coord['x']
    data['y'] = node_coord['y']
    data['z'] = node_coord['z']

In [ ]:
celltypes = list({attr.get('superclass', 'unknown') for _, attr in g.nodes(data=True)})

### Distance Network

In [ ]:
nodes = connectome.nodes()
distance_network = nx.DiGraph()
distance_network.add_nodes_from(nodes)

In [ ]:
for n, data in distance_network.nodes(data=True):
    found = False
    try:
        data['celltype'] = data_s2[data_s2['left_id'] == str(n)]['celltype'].iloc[0]
        data['additional_annotations'] = data_s2[data_s2['left_id'] == str(n)]['additional_annotations'].iloc[0]
        data['level_7_cluster'] = data_s2[data_s2['left_id'] == str(n)]['level_7_cluster'].iloc[0]
        data['hemisphere'] = 'right'
        found = True
    except IndexError:
        pass
    if not found:
        try:
            data['celltype'] = data_s2[data_s2['right_id'] == str(n)]['celltype'].iloc[0]
            data['additional_annotations'] = data_s2[data_s2['right_id'] == str(n)]['additional_annotations'].iloc[0]
            data['level_7_cluster'] = data_s2[data_s2['right_id'] == str(n)]['level_7_cluster'].iloc[0]
            data['hemisphere'] = 'left'
        except IndexError:
            data['celltype'] = 'undefined'
            data['additional_annotations'] = 'undefined'
            data['level_7_cluster'] = 'undefined'
            data['hemisphere'] = 'undefined'
    node_coord = coord[coord['id']==n].iloc[0]
    data['x'] = node_coord['x']
    data['y'] = node_coord['y']
    data['z'] = node_coord['z']

In [ ]:
for celltype in celltypes:
    nodes_celltype = [nodo for nodo, attr in distance_network.nodes(data=True) if attr.get('celltype') == celltype]

    for n1 in nodes_celltype:
        for n2 in nodes_celltype:
            

## Analisi delle due reti

In [ ]:
def basic_info(graph, w=0):
    """
    Analizza un grafo diretto e pesato con diverse metriche:
    - Numero di nodi
    - Numero di archi
    - Degree medio (entrante, uscente, totale)
    - Degree medio ponderato (entrante, uscente, totale)
    - Densità
    - Clustering coefficient medio (non diretto)
    - Lunghezza media dei cammini (pesata)
    - Diametro (pesato)
    - Degree assortativity (entrante-uscente)
    - Massimo componente fortemente connessa
    
    :param graph: Grafo diretto e pesato (NetworkX DiGraph)
    :return: Dizionario con le metriche
    """
    metrics = {}
    
    # Numero di nodi
    metrics['num_nodes'] = graph.number_of_nodes()
    
    # Numero di archi
    metrics['num_edges'] = graph.number_of_edges()

    if w==0:
        # Degree medio (entrante, uscente e totale)
        metrics['average_in_degree'] = sum(dict(graph.in_degree()).values()) / metrics['num_nodes'] if metrics['num_nodes'] > 0 else 0
        metrics['average_out_degree'] = sum(dict(graph.out_degree()).values()) / metrics['num_nodes'] if metrics['num_nodes'] > 0 else 0
        metrics['average_total_degree'] = sum(dict(graph.degree()).values()) / metrics['num_nodes'] if metrics['num_nodes'] > 0 else 0
        metrics['average_clustering'] = nx.average_clustering(graph)
    else:
        # Degree medio ponderato (entrante, uscente e totale)
        weighted_in_degree = sum(dict(graph.in_degree(weight='weight')).values())
        weighted_out_degree = sum(dict(graph.out_degree(weight='weight')).values())
        metrics['average_weighted_in_degree'] = weighted_in_degree / metrics['num_nodes'] if metrics['num_nodes'] > 0 else 0
        metrics['average_weighted_out_degree'] = weighted_out_degree / metrics['num_nodes'] if metrics['num_nodes'] > 0 else 0
        metrics['average_weighted_total_degree'] = sum(dict(graph.degree(weight='weight')).values()) / metrics['num_nodes'] if metrics['num_nodes'] > 0 else 0
        metrics['average_clustering'] = nx.average_clustering(graph, weight='weight')
    
    # Densità del grafo
    metrics['density'] = nx.density(graph)
    
    # Calcolo delle componenti
    if nx.is_strongly_connected(graph):
        # Se il grafo è fortemente connesso
        metrics['average_path_length'] = nx.average_shortest_path_length(graph, weight='weight')
        metrics['diameter'] = nx.diameter(graph, weight='weight')
    else:
        # Analizzare la componente fortemente connessa più grande
        largest_scc = max(nx.strongly_connected_components(graph), key=len)
        subgraph = graph.subgraph(largest_scc)
        if w==0:
            metrics['average_path_length'] = nx.average_shortest_path_length(subgraph)
        else:
            metrics['average_path_length'] = nx.average_shortest_path_length(subgraph, weight='weight')

    metrics['diameter'] = max([max(j.values()) for (i,j) in nx.shortest_path_length(graph)])
    
    # Degree assortativity (entrante-uscente)
    metrics['degree_assortativity'] = nx.degree_assortativity_coefficient(graph)
    
    # Massimo componente fortemente connessa (numero di nodi nella componente più grande)
    largest_scc_size = len(max(nx.strongly_connected_components(graph), key=len))
    metrics['largest_strongly_connected_component'] = largest_scc_size
    
    return metrics

In [ ]:
connectome_metrics = basic_info(connectome)

In [ ]:
connectome_metrics

In [ ]:
directed_network_metrics = basic_info(distance_network)

In [ ]:
directed_network_metrics

# Colonist neurons analysis

In [ ]:
celltypes = list({attr.get('celltype', 'unknown') for _, attr in connectome.nodes(data=True)})
celltypes.remove('undefined')

## Betweenness centrality

In [ ]:
# calcolo della betweenness
connectome_betweenness = nx.betweenness_centrality(connectome)

# prendo i nodi con i più alti valori di betweenness (20%)
num_20 = len(connectome.nodes())*20/100
top_betweenness = heapq.nlargest(int(num_20), connectome_betweenness, key=connectome_betweenness.get)
top_betweenness = [str(node) for node in top_betweenness]

## Closeness centrality

In [ ]:
# calcolo della closeness
connectome_closeness = nx.closeness_centrality(connectome)

# prendo i nodi con i più alti valori di closeness (20%)
top_closeness = heapq.nlargest(int(num_20), connectome_closeness, key=connectome_closeness.get)
top_closeness = [str(node) for node in top_closeness]

## Degree centrality

In [ ]:
# calcolo della betweenness
connectome_degree = nx.degree_centrality(connectome)

# prendo i nodi con i più alti valori di betweenness (20%)
top_degree = heapq.nlargest(int(num_20), connectome_degree, key=connectome_degree.get)
top_degree = [str(node) for node in top_degree]

In [ ]:
for celltype in celltypes:
    nodes = [nodo for nodo, attr in connectome.nodes(data=True) if attr.get('celltype') == celltype]
    with open('Distanze/far_nodes_'+celltype+'_left.pickle', 'rb') as f:
            colonists = pickle.load(f)
    with open('Distanze/far_nodes_'+celltype+'_right.pickle', 'rb') as f:
            colonists.extend(pickle.load(f))
    print(colonists)
    colonists_betweenness = set(top_betweenness).intersection(colonists)
    colonists_closeness = set(top_closeness).intersection(colonists)
    colonists_degree = set(top_degree).intersection(colonists)
    print(celltype + ': \nout of ' + str(len(nodes)) + ' neurons ' + str(len(colonists)) + ' are colonist neurons' )
    print('and out of ' + str(len(colonists)) + ' colonists ' + str(len(colonists_betweennes)) + ' of them belong to the top 20% of betweenness' )
    print('and out of ' + str(len(colonists)) + ' colonists ' + str(len(colonists_closeness)) + ' of them belong to the top 20% of closeness' )
    print('and out of ' + str(len(colonists)) + ' colonists ' + str(len(colonists_degree)) + ' of them belong to the top 20% of degree' )
    print()